<a href="https://colab.research.google.com/github/kkzkwtr/GCI_2025_summer_competition01/blob/main/GCI_competition01_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame, Series
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
path = "/content/drive/My Drive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/"

df = pd.read_csv(path + 'data/train.csv')
df_test = pd.read_csv(path + 'data/test.csv')

#??????
df_concat = pd.concat([df,df_test],ignore_index=True)

In [ ]:
print('???????????{}????{}?????'.format(df.shape[0], df.shape[1]))
print('????????????{}????{}????'.format(df_test.shape[0], df_test.shape[1]))

In [ ]:
print(df_concat[:891].shape, df_concat[891:].shape)

In [ ]:
df_concat.info()

In [ ]:
df.info()

###????'Age','Cabin','Embarked'

In [ ]:
from sklearn.ensemble import RandomForestRegressor
#Age?Pclass, Sex, Parch, SibSp?????????????

# ??: https://qiita.com/jun40vn/items/d8a1f71fae680589e05c

#????
age_df_concat = df_concat[["Age", "Pclass", "Sex", "Parch", "SibSp"]]

age_df_concat.replace({'Sex': {'male': 0, 'female': 1}}, inplace=True)

#?????->????????????????->?????
known_age = age_df_concat[age_df_concat.Age.notnull()].values
unknown_age = age_df_concat[age_df_concat.Age.isnull()].values

X = known_age[:, 1:]
y = known_age[:, 0]

# ????????????????
rf_age = RandomForestRegressor(n_estimators=100,random_state=42, n_jobs=-1)
rf_age.fit(X, y)

predictedAges = rf_age.predict(unknown_age[:, 1:])
df_concat.loc[ (df_concat.Age.isnull()), 'Age' ] = predictedAges

In [ ]:
df_concat.isnull().sum()

In [ ]:
#????????????
facet = sns.FacetGrid(df_concat[:891], hue='Perished',aspect=2)
facet.map(sns.kdeplot, 'Age', shade=True)
facet.set(xlim=(0, df_concat[:891]['Age'].max()))
facet.add_legend()
plt.show()

In [ ]:
#Cabin????
df_concat['Cabin'] = df_concat['Cabin'].fillna('Unknown')
df_concat['Cabin_label'] = df_concat['Cabin'].str.get(0)
mean_perished = df_concat.groupby('Cabin_label')['Perished'].mean()
order = mean_perished.sort_values(ascending=True).index.tolist()
sns.barplot(x='Cabin_label', y='Perished', data=df_concat,order=order, palette='Set3')
plt.show()

In [ ]:
# Cabin_label???????(????)
df_concat.loc[(df_concat['Cabin_label'] == 'T') | (df_concat['Cabin_label'] == 'U') | (df_concat['Cabin_label'] == 'A'), 'Cabin_label'] = 2
df_concat.loc[(df_concat['Cabin_label'] == 'G') | (df_concat['Cabin_label'] == 'C') | (df_concat['Cabin_label'] == 'F'), 'Cabin_label'] = 1
df_concat.loc[(df_concat['Cabin_label'] == 'B') | (df_concat['Cabin_label'] == 'D') | (df_concat['Cabin_label'] == 'E'), 'Cabin_label'] = 0
df_concat.drop(['Cabin'], axis=1, inplace=True)
df_concat.info()

In [ ]:
df_concat.info()

In [ ]:
#Embarked???
sns.countplot(x='Embarked', data=df_concat, palette='Set3')
plt.show()

In [ ]:
df_concat['Embarked'].fillna('S', inplace=True)

df_concat.isnull().sum()

In [ ]:
#Embarked?Pclass??????(Embarked=S?Pclass=3)?Fare?????
fare = df_concat[(df_concat['Embarked'] == 'S')&(df_concat['Pclass'] == 3) ]['Fare'].median()
df_concat['Fare'].fillna(fare, inplace=True)
df_concat.isnull().sum()

In [ ]:
df_concat.info()

In [ ]:
#Sex?one-hot?
# df_concat.replace({'Sex': {'male': 0, 'female': 1}},inplace=True)
sex = df_concat['Sex']
sex_ohe = pd.get_dummies(sex,prefix='Sex')
df_concat = pd.concat([df_concat, sex_ohe], axis=1)
df_concat.drop(['Sex'], axis=1, inplace=True)
df_concat.info()

In [ ]:
#Ticket??
Ticket_counts = dict(df_concat['Ticket'].value_counts())
df_concat['TicketGroup'] = df_concat['Ticket'].map(Ticket_counts)
sns.barplot(x='TicketGroup', y='Perished', data=df_concat, palette='Set3')
# sns.countplot(x='TicketGroup', data=df, palette='Set3')
plt.show()

# ?????????????????
# df['is_Ticket_Prefix'] = df['Ticket'].str.contains(r'[a-zA-Z]')
# # df_concat['is_Ticket_Prefix'] = df_concat['Ticket'].str.contains(r'[a-zA-Z]')
# sns.barplot(x='is_Ticket_Prefix', y='Perished', data=df, palette='Set3')
# plt.show()

In [ ]:
# Ticket???????
df_concat.loc[(df_concat['TicketGroup'] >= 2) & (df_concat['TicketGroup'] <= 4), 'Ticket_label'] = 2
df_concat.loc[(df_concat['TicketGroup'] >= 5) & (df_concat['TicketGroup'] <= 8) | (df_concat['TicketGroup']==1), 'Ticket_label'] = 1
df_concat.loc[(df_concat['TicketGroup'] >=11), 'Ticket_label'] = 0
df_concat.drop(['TicketGroup','Ticket'], axis=1, inplace=True)
df_concat.info()

In [ ]:
sns.barplot(x='Ticket_label', y='Perished', data=df_concat, palette='Set3')
plt.show()

In [ ]:
df_concat.isnull().sum()

In [ ]:
#Name????????????
df_concat['Title'] = df_concat['Name'].map(lambda x: x.split(',')[1].split('.')[0].strip())
df_concat['Title'].value_counts()

In [ ]:
df_concat['Title'].replace(['Rev','Dr', 'Col', 'Major', 'Capt'], 'Officer', inplace=True)
df_concat['Title'].replace(['Don', 'Sir', 'the Countess', 'Lady', 'Dona'], 'Royalty', inplace=True)
df_concat['Title'].replace(['Mme', 'Ms', 'Mrs'], 'Mrs', inplace=True)
df_concat['Title'].replace(['Mlle', 'Miss'], 'Miss', inplace=True)
df_concat['Title'].replace(['Mr'], 'Mr', inplace=True)
df_concat['Title'].replace(['Jonkheer'],'Master', inplace=True)
sns.barplot(x='Title', y='Perished', data=df_concat[:891], palette='Set3')
plt.show()

In [ ]:
#Name,Ticket?????
df_concat.drop(['Name'], axis=1, inplace=True)
df_concat.info()

In [ ]:
#embarked?one-hot?
emberked = df_concat['Embarked']
emberked_ohe = pd.get_dummies(emberked,prefix='Embarked')
df_concat = pd.concat([df_concat, emberked_ohe], axis=1)
df_concat.drop(['Embarked'], axis=1, inplace=True)
df_concat.info()

In [ ]:
cabin_label = df_concat['Cabin_label']
cabin_ohe = pd.get_dummies(cabin_label,prefix='Cabin')
df_concat = pd.concat([df_concat, cabin_ohe], axis=1)
df_concat.drop(['Cabin'], axis=1, inplace=True)
df_concat.drop(['Cabin_label'], axis=1, inplace=True)
df_concat.info()

In [ ]:
#Family????? + ??????
#https://qiita.com/jun40vn/items/d8a1f71fae680589e05c
df_concat['Family'] = df_concat['SibSp'] + df_concat['Parch'] + 1
df_concat.loc[(df_concat['Family'] >= 2) & (df_concat['Family'] <= 4), 'Family_label'] = 2
df_concat.loc[(df_concat['Family'] >= 5) & (df_concat['Family'] <= 7) | (df_concat['Family']==1), 'Family_label'] = 1
df_concat.loc[(df_concat['Family'] >= 8), 'Family_label'] = 0
df_concat.drop(['Family','SibSp','Parch'], axis=1, inplace=True)
df_concat.info()

In [ ]:
Title = df_concat['Title']
Title_ohe = pd.get_dummies(Title,prefix='Title')
df_concat = pd.concat([df_concat, Title_ohe], axis=1)
df_concat.drop(['Title'], axis=1, inplace=True)
df_concat.info()

In [ ]:
df = df_concat[:891]
df_test = df_concat[891:]
df_test.isnull().sum()

In [ ]:
X = df.drop(['PassengerId', 'Perished'], axis=1).values
y = df['Perished'].values
X_test = df_test.drop(['PassengerId', 'Perished'], axis=1).values

In [ ]:
X.shape, y.shape, X_test.shape

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
# prompt: LightGBM?????????????????????????????

import pandas as pd
import matplotlib.pyplot as plt
!pip install lightgbm
import lightgbm as lgb

# LightGBM??????
lgb_clf = lgb.LGBMClassifier(random_state=42)
lgb_clf.fit(X_train, y_train)

# ?????????
importance = lgb_clf.feature_importances_
feature_names = df.drop(['PassengerId', 'Perished'], axis=1).columns

# DataFrame????????
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
importance_df = importance_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance_df)
plt.title('LightGBM Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()


In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None,3, 5],
    'min_samples_split': [2, 3, 10],
    'min_samples_leaf': [3, 10]
}

grid_rf = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy')
grid_rf.fit(X, y)

print("Best parameters: ", grid_rf.best_params_)
print('CV Score: ', grid_rf.best_score_)

In [ ]:
rf = RandomForestClassifier(
    max_depth=None,n_estimators=100,min_samples_leaf=3,min_samples_split=2,
    random_state=42, n_jobs=-1
)

# ????20????
# from sklearn.feature_selection import SelectKBest
# from sklearn.pipeline import make_pipeline
# select = SelectKBest(k = 20)

# pipe = make_pipeline(select, rf)
# pipe.fit(X_train, y_train)
rf.fit(X_train, y_train)
print('Train Score: ', rf.score(X_train, y_train))
print('Valid Score: ', rf.score(X_valid, y_valid))

In [ ]:
lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)
print('Train Score: ', lr.score(X_train, y_train))
print('Valid Score: ', lr.score(X_valid, y_valid))

In [ ]:
mlpc = MLPClassifier(hidden_layer_sizes=(100, 100 ,10),random_state=42)
mlpc.fit(X_train, y_train)
print('Train Score: ', mlpc.score(X_train, y_train))
print('Valid Score: ', mlpc.score(X_valid, y_valid))

In [ ]:
#XGBoost
import xgboost as xgb

xgb_find = xgb.XGBClassifier(random_state=42)
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'min_samples_split': [2, 3, 10],
    'min_samples_leaf': [1, 3, 10]
}

grid_xgb = GridSearchCV(xgb_find, param_grid, cv=5, scoring='accuracy')
grid_xgb.fit(X, y)

print("Best parameters: ", grid_xgb.best_params_)
print('CV Score: ', grid_xgb.best_score_)


In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    max_depth=3,n_estimators=100,min_samples_leaf=1,min_samples_split=2,
    random_state=42, n_job=-1,
)

xgb_model.fit(X_train, y_train)
print('Train Score: ', xgb_model.score(X_train, y_train))
print('Valid Score: ', xgb_model.score(X_valid, y_valid))

In [ ]:
# GBC
from sklearn.ensemble import GradientBoostingClassifier

gbc_find = GradientBoostingClassifier(random_state=42)
param_grid = {
    'n_estimators': [100],
    'max_depth': [3, 5],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 3],
    'learning_rate': [0.01, 0.1],
    'validation_fraction': [0.1, 0.2]
}

grid_gbc = GridSearchCV(gbc_find, param_grid, cv=5, scoring='accuracy')
grid_gbc.fit(X, y)

print("Best parameters: ", grid_gbc.best_params_)
print('CV Score: ', grid_gbc.best_score_)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(
    max_depth=3,n_estimators=100,min_samples_leaf=3,min_samples_split=2,learning_rate=0.1,
    validation_fraction=0.1,
    random_state=42,
)

gbc.fit(X_train, y_train)
print('Train Score: ', gbc.score(X_train, y_train))
print('Valid Score: ', gbc.score(X_valid, y_valid))

In [ ]:
#SVC
from sklearn.svm import SVC
svc = SVC(kernel= 'rbf',C=1.0, gamma='scale',probability=True,random_state=42)
svc.fit(X_train, y_train)
print('Train Score: ', svc.score(X_train, y_train))
print('Valid Score: ', svc.score(X_valid, y_valid))

In [ ]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('rfc', rf),
        ('gbc', gbc),
        ('lr', lr),
        ('xgb', xgb_model),
    ],
    voting='soft',
)

voting.fit(X_train, y_train)
print('Train Score: ', voting.score(X_train, y_train))
print('Valid Score: ', voting.score(X_valid, y_valid))

In [ ]:
#??????
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[
        ('rfc', rf),
        ('gbc', gbc),
        ('lr', lr),
        ('xgb',xgb_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
)

stacking.fit(X_train, y_train)
print('Train Score: ', stacking.score(X_train, y_train))
print('Valid Score: ', stacking.score(X_valid, y_valid))

In [ ]:
#VotingClassifier??????
voting_pred = voting.predict_proba(X_test)
pred = voting_pred.argmax(axis=1)

In [ ]:
stacking_pred = stacking.predict_proba(X_test)
pred = stacking_pred.argmax(axis=1)

In [ ]:
#selectKBest
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import make_pipeline

select = SelectKBest(k = 20)

pipe = make_pipeline(select, rf)
pipe.fit(X_train, y_train)
rfc_pred = pipe.predict_proba(X_test)
pred = rfc_pred.argmax(axis=1)

In [ ]:
df.columns

In [ ]:
mask = select.get_support()
list_col = list(df.columns[2:])
for i, j in enumerate(list_col):
    print('No'+str(i+1), j,'=', mask[i])
X_selected = select.transform(X)
print('X.shape={}, X_selected.shape={}'.format(X.shape, X_selected.shape))

In [ ]:
#????????????csv
rfc_pred = rf.predict_proba(X_test)
pred = rfc_pred.argmax(axis=1)

In [ ]:
rfc_pred = rf.predict_proba(X_test)
lr_pred = lr.predict_proba(X_test)
mlpc_pred = mlpc.predict_proba(X_test)

pred_proba = (rfc_pred + lr_pred + mlpc_pred) / 3
pred = pred_proba.argmax(axis=1)

In [ ]:
path = '/content/drive/My Drive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/'

submission = pd.read_csv(path + 'gender_submission.csv')
submission

In [ ]:
pred.shape

In [ ]:
submission['Perished'] = pred
submission

In [ ]:
# JupyterLab?????????????
# ????????????????
submission.to_csv('/content/drive/My Drive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/submission_v10.csv', index=False)

In [ ]:
test_augumented = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/data/test_augmented.csv')
test_augumented.info()

In [ ]:
# prompt: test_augumented?Survived????Perished????0,1?????????
test_augumented = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/data/test_augmented.csv')
test_augumented.rename(columns={'Survived': 'Perished'}, inplace=True)
test_augumented['Perished'] = 1 - test_augumented['Perished']

In [ ]:
#????????
from sklearn.metrics import accuracy_score
accuracy_score(submission['Perished'], test_augumented['Perished'])

In [ ]:
# prompt: google drive??submission_v{??}.csv???????????????csv?perished??accuracy_score???????????

import pandas as pd
import os

# Google Drive??submission???????
submission_dir = '/content/drive/My Drive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/'

# ????????????????
test_augmented_path = '/content/drive/MyDrive/Colab Notebooks/GCI_2025_summer/Competitions/competition_1/data/test_augmented.csv'
test_augmented = pd.read_csv(test_augmented_path)
test_augmented.rename(columns={'Survived': 'Perished'}, inplace=True)
test_augmented['Perished'] = 1 - test_augmented['Perished'] # ??????Perished?????

# submission_v{??}.csv ?????????accuracy_score???
accuracy_scores = {}
for file in os.listdir(submission_dir):
    if file.startswith('submission_v') and file.endswith('.csv'):
        try:
            submission_path = os.path.join(submission_dir, file)
            submission_df = pd.read_csv(submission_path)

            # submission?????test_augmented?????PassengerId??????????
            if submission_df['PassengerId'].equals(test_augmented['PassengerId']):
                accuracy = accuracy_score(test_augmented['Perished'], submission_df['Perished'])
                accuracy_scores[file] = accuracy
            else:
                print(f"Warning: PassengerId mismatch between {file} and test_augmented.csv. Skipping.")

        except Exception as e:
            print(f"Error processing {file}: {e}")

# accuracy_score????????????
print("Accuracy scores for submission files:")
for file, score in sorted(accuracy_scores.items()): # ?????????????
    print(f"{file}: {score:.4f}")
